Preparing the PAAL ADL dataset (Climent-Perez_Munoz-Anton_Poli, 2022) for the analysis. Dataset downloaded from: https://zenodo.org/records/5785955

In [ ]:
import os
import warnings
from pathlib import Path
import pandas as pd

# Suppress warnings from Python and external libraries
os.environ["PYTHONWARNINGS"] = "ignore"  # Suppress Python warnings
warnings.filterwarnings("ignore")

In [ ]:
fs = 32  # Hz, from PAAL ADL dataset description (Climent-Perez_Munoz-Anton_Poli, 2022)
time_per_sample = 1000 / fs  # milliseconds per sample

In [ ]:
metadata_df = pd.read_csv('../Data/PAAL_ADL/Raw/ADLs.csv', header=None)
metadata_df

,0,1
0,1,drink water
1,2,eat meal
2,3,open bottle
3,4,open a box
4,5,brush teeth
5,6,brush hair
6,7,take off jacket
7,8,put on jacket
8,9,put on a shoe
9,10,take off a shoe


In [ ]:
activity_dict = dict(zip(metadata_df[0], metadata_df[1]))
activity_dict

{1: 'drink water',
 2: 'eat meal',
 3: 'open bottle',
 4: 'open a box',
 5: 'brush teeth',
 6: 'brush hair',
 7: 'take off jacket',
 8: 'put on jacket',
 9: 'put on a shoe',
 10: 'take off a shoe',
 11: 'put on glasses',
 12: 'take off glasses',
 13: 'sit down',
 14: 'stand up',
 15: 'writing',
 16: 'phone call',
 17: 'type on a keyboard',
 18: 'salute',
 19: 'sneeze/cough',
 20: 'blow nose',
 21: 'washing hands',
 22: 'dusting',
 23: 'ironing',
 24: 'washing dishes'}

In [ ]:
# Correcting the activity_dict to match the data names in the UiS4ADL dataset
activity_dict[3] = 'open a bottle'
activity_dict[7] = 'take off a jacket'
activity_dict[8] = 'put on a jacket'

In [ ]:
data_dir_path = Path('../Data/PAAL_ADL/Raw/dataset')

# Iterate through all files in the directory
for file_path in data_dir_path.rglob("*.csv"):
    if file_path.is_file():
        print(f"File: {file_path}")
    elif file_path.is_dir():
        print(f"Directory: {file_path}")

File: ../Data/PAAL_ADL/Raw/dataset/take_off_a_jacket_45_1.csv
File: ../Data/PAAL_ADL/Raw/dataset/salute_37_3.csv
File: ../Data/PAAL_ADL/Raw/dataset/brush_hair_30_1.csv
File: ../Data/PAAL_ADL/Raw/dataset/brush_hair_5_1.csv
File: ../Data/PAAL_ADL/Raw/dataset/take_off_a_shoe_10_0.csv
File: ../Data/PAAL_ADL/Raw/dataset/put_on_a_shoe_25_1.csv
File: ../Data/PAAL_ADL/Raw/dataset/type_on_a_keyboard_43_4.csv
File: ../Data/PAAL_ADL/Raw/dataset/take_off_a_jacket_26_2.csv
File: ../Data/PAAL_ADL/Raw/dataset/eat_meal_38_3.csv
File: ../Data/PAAL_ADL/Raw/dataset/dusting_49_3.csv
File: ../Data/PAAL_ADL/Raw/dataset/take_off_a_shoe_36_3.csv
File: ../Data/PAAL_ADL/Raw/dataset/blow_nose_32_4.csv
File: ../Data/PAAL_ADL/Raw/dataset/salute_11_0.csv
File: ../Data/PAAL_ADL/Raw/dataset/brush_hair_16_2.csv
File: ../Data/PAAL_ADL/Raw/dataset/take_off_glasses_6_3.csv
File: ../Data/PAAL_ADL/Raw/dataset/put_on_a_shoe_46_2.csv
File: ../Data/PAAL_ADL/Raw/dataset/sit_down_5_2.csv
File: ../Data/PAAL_ADL/Raw/dataset/washi

## Combine all files into one PAAL_ADL dataset

In [ ]:
# Reverse activity_dict for quick name lookup
activity_name_to_id = {v.replace("/", " "): k for k, v in activity_dict.items()}

all_data = []
file_id = 1

In [ ]:
activity_name_to_id

{'drink water': 1,
 'eat meal': 2,
 'open a bottle': 3,
 'open a box': 4,
 'brush teeth': 5,
 'brush hair': 6,
 'take off a jacket': 7,
 'put on a jacket': 8,
 'put on a shoe': 9,
 'take off a shoe': 10,
 'put on glasses': 11,
 'take off glasses': 12,
 'sit down': 13,
 'stand up': 14,
 'writing': 15,
 'phone call': 16,
 'type on a keyboard': 17,
 'salute': 18,
 'sneeze cough': 19,
 'blow nose': 20,
 'washing hands': 21,
 'dusting': 22,
 'ironing': 23,
 'washing dishes': 24}

In [ ]:
# Read an example file to analyse its structure
example_file_path = "../Data/PAAL_ADL/Raw/dataset/take_off_a_jacket_45_1.csv"
example_file = pd.read_csv(example_file_path, header=None) # PAAL ADL contains only accelometer data, given in raw counts
example_file

,0,1,2
0,-76.0,8.0,-40.0
1,-55.0,7.0,-34.0
2,-53.0,4.0,-32.0
3,-54.0,5.0,-27.0
4,-64.0,8.0,-29.0
...,...,...,...
194,-74.0,-16.0,5.0
195,-58.0,-20.0,-5.0
196,-64.0,-16.0,-12.0
197,-71.0,-9.0,-27.0


In [ ]:
# Read all the CSV files and collect accelerometer data into one dataframe
# Create a timestamp column and add metadata (adl, session, subject) from the filename. 
# Also create file_id column which is incremented for each file.
for file_path in data_dir_path.rglob("*.csv"):
    if file_path.is_file():
        filename = file_path.stem  # without .csv
        parts = filename.split('_')
        
        # Extract metadata
        subject = int(parts[-2])
        session = int(parts[-1])
        adl_name = " ".join(parts[:-2])
        adl_id = activity_name_to_id[adl_name]
        print(f"ADL: '{adl_name}' ({adl_id}), Subject: {subject}, Session: {session}", end="\n\n")
        
        # Read CSV files containing raw accelerometer data in counts
        # Empatica E4 uses ±2g range with 64 counts = 1g conversion factor 
        # (from: https://www.sciencedirect.com/science/article/pii/S2352340922001081)
        data_df = pd.read_csv(file_path, header=None, names=['accX[counts]', 'accY[counts]', 'accZ[counts]'])
        
        # Convert counts to milligravity (mg)
        # 1 count = 1g / 64 = 1000mg / 64 = 15.625 mg/count
        COUNTS_TO_MG = 1000 / 64  # = 15.625 mg/count
        
        data_df['accX[mg]'] = data_df['accX[counts]'] * COUNTS_TO_MG
        data_df['accY[mg]'] = data_df['accY[counts]'] * COUNTS_TO_MG
        data_df['accZ[mg]'] = data_df['accZ[counts]'] * COUNTS_TO_MG
        
        # Drop the count columns
        data_df = data_df.drop(columns=['accX[counts]', 'accY[counts]', 'accZ[counts]'])
        
        # Add derived columns
        data_df['timestamp'] = (data_df.index * time_per_sample).astype(float)
        data_df['adl'] = adl_id
        data_df['session'] = session
        data_df['subject'] = subject
        data_df['fileID'] = file_id
        
        all_data.append(data_df)
        file_id += 1

ADL: 'take off a jacket' (7), Subject: 45, Session: 1

ADL: 'salute' (18), Subject: 37, Session: 3

ADL: 'brush hair' (6), Subject: 30, Session: 1

ADL: 'brush hair' (6), Subject: 5, Session: 1

ADL: 'take off a shoe' (10), Subject: 10, Session: 0

ADL: 'put on a shoe' (9), Subject: 25, Session: 1

ADL: 'type on a keyboard' (17), Subject: 43, Session: 4

ADL: 'take off a jacket' (7), Subject: 26, Session: 2

ADL: 'eat meal' (2), Subject: 38, Session: 3

ADL: 'dusting' (22), Subject: 49, Session: 3

ADL: 'take off a shoe' (10), Subject: 36, Session: 3

ADL: 'blow nose' (20), Subject: 32, Session: 4

ADL: 'salute' (18), Subject: 11, Session: 0

ADL: 'brush hair' (6), Subject: 16, Session: 2

ADL: 'take off glasses' (12), Subject: 6, Session: 3

ADL: 'put on a shoe' (9), Subject: 46, Session: 2

ADL: 'sit down' (13), Subject: 5, Session: 2

ADL: 'washing hands' (21), Subject: 9, Session: 3

ADL: 'ironing' (23), Subject: 25, Session: 0

ADL: 'phone call' (16), Subject: 36, Session: 4

ADL:

In [ ]:
final_df = pd.concat(all_data, ignore_index=True)
final_df = final_df[['timestamp', 'accX[mg]', 'accY[mg]', 'accZ[mg]', 'adl', 'session', 'subject', 'fileID']] # Reorder columns

final_df

,timestamp,accX[mg],accY[mg],accZ[mg],adl,session,subject,fileID
0,0.00,-1187.500,125.000,-625.000,7,1,45,1
1,31.25,-859.375,109.375,-531.250,7,1,45,1
2,62.50,-828.125,62.500,-500.000,7,1,45,1
3,93.75,-843.750,78.125,-421.875,7,1,45,1
4,125.00,-1000.000,125.000,-453.125,7,1,45,1
...,...,...,...,...,...,...,...,...
1432168,3500.00,-156.250,-812.500,343.750,18,4,24,6072
1432169,3531.25,109.375,-828.125,515.625,18,4,24,6072
1432170,3562.50,62.500,-750.000,265.625,18,4,24,6072
1432171,3593.75,0.000,-703.125,218.750,18,4,24,6072


In [ ]:
# Save the DataFrame after cleaning the inactive movements
output_file = f"../Data/PAAL_ADL/Processed/PAAL_ADL_{fs}hz.csv"

# Remove the dataset file if it already exists
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"{output_file} has been removed successfully.")
else:
    print(f"{output_file} does not exist.")

# Save processed data
final_df.to_csv(output_file, index=False)
print(f"The dataset was saved to {output_file}")

../Data/PAAL_ADL/Processed/PAAL_ADL_32hz.csv does not exist.
The dataset was saved to ../Data/PAAL_ADL/Processed/PAAL_ADL_32hz.csv
